# Weather conditions and injury severity in France (ONISR 2018)

This descriptive analysis studies weather, hour and road user category. Each row represents a road user involved in a recorded injury accident, not a distinct accident or a journey. Severe means killed or hospitalised (grav = 2 or 3). Associations do not establish causation or risk per journey.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
carac = pd.read_csv("ACC-caracteristiques-2018.csv", sep=",", encoding="utf-8", low_memory=False)
lieux = pd.read_csv("ACC-lieux-2018.csv", sep=",", encoding="utf-8", low_memory=False)
usagers = pd.read_csv("ACC-usagers-2018.csv", sep=",", encoding="utf-8", low_memory=False)
vehicules = pd.read_csv("ACC-vehicules-2018.csv", sep=",", encoding="utf-8", low_memory=False)

In [ ]:
print("carac columns:")
print(carac.columns.tolist())

print("usagers columns:")
print(usagers.columns.tolist())

print("lieux columns:")
print(lieux.columns.tolist())

In [ ]:
# Preserve one row per road user; reject duplicate accident keys.
df = usagers[["Num_Acc", "grav", "an_nais", "catu"]].merge(
    carac[["Num_Acc", "atm", "hrmn"]], on="Num_Acc", how="left", validate="many_to_one"
)
df = df.merge(lieux[["Num_Acc", "vosp"]], on="Num_Acc", how="left", validate="many_to_one")
assert len(df) == len(usagers)
print(f"{len(df):,} road users; {df.Num_Acc.nunique():,} distinct accidents")


In [ ]:
hhmm = pd.to_numeric(df["hrmn"], errors="coerce")
valid_time = hhmm.between(0, 2359) & hhmm.mod(100).lt(60) & hhmm.mod(1).eq(0)
df["hour"] = (hhmm // 100).where(valid_time)


In [ ]:
df["age"] = 2018 - df["an_nais"]
df.loc[~df["age"].between(0, 110), "age"] = np.nan
valid_severity = df["grav"].isin([1, 2, 3, 4])
df["severe"] = df["grav"].isin([2, 3])
df_weather = df.loc[valid_severity & df["atm"].isin(range(1, 10))].copy()
print(f"Excluded {len(df) - len(df_weather):,} users with missing/invalid severity or weather.")
print(f"Excluded from hourly analysis: {df_weather.hour.isna().sum():,} users with invalid time.")


In [ ]:
df_weather["catu"] = df_weather["catu"].astype("Int64")

In [ ]:
#labels

atm_labels = {
    1: "Clear",
    2: "Light rain",
    3: "Heavy rain",
    4: "Snow / hail",
    5: "Fog / smoke",
    6: "Strong wind",
    7: "Dazzling weather",
    8: "Overcast",
    9: "Other"
}

catu_labels = {
    1: "Driver",
    2: "Passenger",
    3: "Pedestrian"
}

df_weather["atm_label"] = df_weather["atm"].map(atm_labels).fillna("Unknown")
df_weather["catu_label"] = df_weather["catu"].map(catu_labels).fillna("Unknown")

## 1. Severe injury proportion by weather

`n` counts road users, including uninjured people recorded in these accidents. It does not count distinct accidents.


In [ ]:
weather_analysis = (
    df_weather.groupby("atm_label")
    .agg(
        n=("Num_Acc", "size"),
        severe_rate=("severe", "mean")
    )
    .sort_values("n", ascending=False)
)

print(weather_analysis)

In [ ]:
labels = [
    f"{cond}\n(n={weather_analysis.loc[cond, 'n']})"
    for cond in weather_analysis.index
]

plt.figure(figsize=(10,5))

sns.barplot(
    x=labels,
    y=weather_analysis["severe_rate"]
)

plt.title("Severe injury proportion by weather")
plt.ylabel("Proportion of road users killed or hospitalised")

plt.xticks(rotation=45)

plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## 2. Severe injury proportion by hour


In [ ]:
hour_analysis = df_weather.groupby("hour").agg(n=("Num_Acc", "size"), severe_rate=("severe", "mean")).reset_index()
print(hour_analysis.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(hour_analysis["hour"], hour_analysis["severe_rate"], marker="o")
for row in hour_analysis.itertuples():
    ax.annotate(f"n={row.n:,}", (row.hour, row.severe_rate), xytext=(0, 8), textcoords="offset points", ha="center", rotation=60, fontsize=8)
ax.set(title="Severe injury proportion among recorded road users, by hour", xlabel="Hour of day", ylabel="Proportion killed or hospitalised")
ax.set_xticks(range(24))
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


## 3. Severe injury proportion by road user category


In [ ]:
user_analysis = df_weather.loc[df_weather["catu"].isin([1, 2, 3])].groupby("catu").agg(n=("Num_Acc", "size"), severe_rate=("severe", "mean")).reset_index()
user_analysis["catu_label"] = user_analysis["catu"].map(catu_labels)
print(user_analysis.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
labels = [f"{r.catu_label}\n(n={r.n:,})" for r in user_analysis.itertuples()]
ax.bar(labels, user_analysis["severe_rate"])
ax.set(title="Severe injury proportion by road user category", xlabel="Road user category", ylabel="Proportion killed or hospitalised", ylim=(0, 1))
plt.tight_layout()
plt.show()


## 4. Weather and road user category

Labels show the number of users (n). Small groups are less stable; n < 100 is flagged for attention, not used as a statistical significance threshold. Users from the same accident are not independent.


In [ ]:
df_user_weather = df_weather.loc[df_weather["catu"].isin([1, 2, 3])].copy()
user_weather_analysis = df_user_weather.groupby(["atm_label", "catu_label"]).agg(n=("Num_Acc", "size"), severe_rate=("severe", "mean")).reset_index()
user_weather_analysis["small_group"] = user_weather_analysis["n"] < 100
main_weather = ["Clear", "Light rain", "Fog / smoke", "Overcast"]
filtered_user_weather = user_weather_analysis.loc[user_weather_analysis["atm_label"].isin(main_weather)]
print(filtered_user_weather.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
user_order = ["Driver", "Passenger", "Pedestrian"]
x = np.arange(len(main_weather))
width = 0.25
for j, category in enumerate(user_order):
    values = filtered_user_weather.loc[filtered_user_weather.catu_label.eq(category)].set_index("atm_label").reindex(main_weather)
    positions = x + (j - 1) * width
    ax.bar(positions, values["severe_rate"], width, label=category)
    for pos, row in zip(positions, values.itertuples()):
        if pd.notna(row.n):
            ax.text(pos, row.severe_rate + 0.015, f"n={int(row.n):,}" + ("*" if row.n < 100 else ""), ha="center", fontsize=8, rotation=45)
ax.set_xticks(x, main_weather)
ax.set(title="Severe injury proportion by weather and road user category", xlabel="Weather (* n < 100)", ylabel="Proportion killed or hospitalised", ylim=(0, 1))
ax.legend(title="Road user")
plt.tight_layout()
plt.show()


## Interpretation and limitations

Compare proportions together with group sizes. These are proportions among people recorded in injury accidents, including uninjured involved users; they are not accident frequencies or risks per trip. Weather, time, road type and other factors may be related. No causal effect is established. In particular, interpret the fog/pedestrian subgroup cautiously and report its count. A stronger analysis would account for shared accident IDs and potential confounding factors.


## Results from the supplied 2018 data

- Fog/smoke: 32.4% killed or hospitalised among 945 users; clear weather: 19.5% among 103,511 users.
- The highest hourly proportion is at 04:00–04:59: 30.5% among 1,177 users.
- Pedestrians: 36.0% among 10,960 users; drivers: 18.1% among 96,017; passengers: 18.4% among 23,183.
- Pedestrians in fog/smoke: 54.9%, but only 51 users (28 killed or hospitalised). This small subgroup warrants caution.

These descriptive results concern recorded users, not the probability of an accident per journey. Re-run all cells to regenerate the tables and charts.
